In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
corpus = """
Once upon a time, there was a beautiful village surrounded by green mountains.
The villagers lived happily and enjoyed the beauty of nature.
Every morning, the birds sang sweet songs and the sun
spread its golden light across the fields.

A young boy named Alex loved exploring the forests.
One day, he discovered a mysterious path behind an old tree.
He followed the path and found a small wooden house.
Inside the house, there was a magical book filled with stories.

The book told him about distant kingdoms, brave warriors,
hidden treasures and powerful dragons.
Alex was excited to begin his adventure.
He packed some food, took his bag and started his journey.

After walking for many hours, he reached a beautiful river.
The water was clear and the flowers were colorful.
He sat under a tree and watched the sunset.
He knew that this was only the beginning of a wonderful journey.
"""

In [ ]:
# Preprocessing
corpus = corpus.lower().replace("\n", " ")
tokenizer = Tokenizer(char_level=True)  # character-level
tokenizer.fit_on_texts([corpus])
total_chars = len(tokenizer.word_index) + 1

In [ ]:
# Convert text to sequence of integers
encoded = tokenizer.texts_to_sequences([corpus])[0]

In [ ]:
# Prepare input-output sequences
seq_length = 40
sequences = []
for i in range(seq_length, len(encoded)):
    seq = encoded[i-seq_length:i]
    label = encoded[i]
    sequences.append((seq, label))

In [ ]:
X = np.array([s[0] for s in sequences])
y = to_categorical([s[1] for s in sequences], num_classes=total_chars)

In [ ]:
from tensorflow.keras.layers import SimpleRNN # Import SimpleRNN

# Build LSTM/RNN model
model = Sequential([
    Embedding(total_chars, 50, input_length=seq_length),
    # LSTM(128, return_sequences=False),
    SimpleRNN(128, return_sequences=False), # Using SimpleRNN here
    Dense(total_chars, activation='softmax')
])

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(X, y, epochs=40, verbose=1)

Epoch 1/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.1484 - loss: 3.1098
Epoch 2/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1743 - loss: 2.9410
Epoch 3/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1802 - loss: 2.8863
Epoch 4/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1991 - loss: 2.8115
Epoch 5/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2603 - loss: 2.6977
Epoch 6/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2756 - loss: 2.5376
Epoch 7/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3333 - loss: 2.3677
Epoch 8/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3934 - loss: 2.2033
Epoch 9/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4229 - loss: 2.0734
Epoch 10/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4547 - loss: 1.9503
Epoch 11/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.4971 - loss: 1.8279
Epoch 12/40
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy:

In [ ]:
# Function to generate text
def generate_text(model, tokenizer, seq_length, seed_text, n_chars):
    result = []
    in_text = seed_text.lower()
    for _ in range(n_chars):
        encoded = tokenizer.texts_to_sequences([in_text])[-1]
        encoded = encoded[-seq_length:]  # last seq_length chars
        encoded = tf.keras.preprocessing.sequence.pad_sequences([encoded], maxlen=seq_length, padding='pre')
        y_pred = np.argmax(model.predict(encoded, verbose=0))
        out_char = ''
        for char, index in tokenizer.word_index.items():
            if index == y_pred:
                out_char = char
                break
        in_text += out_char
        result.append(out_char)
    return seed_text + ''.join(result)

In [ ]:
# Generate text
print(generate_text(model, tokenizer, seq_length, "Once upon a time", 100))

Once upon a timee fored toere wur ana ans s. erend ansl alorengohed ater sored s hrred wet aisunte biled erima noure
